<a href="https://colab.research.google.com/github/Birkity/10_Academy/blob/Week_0/Fine_Tuning_LeanChems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets peft trl bitsandbytes torch PyPDF2 sentence-transformers nltk faiss-cpu

In [2]:
# Step 1: Setup Environment
!pip install --upgrade trl transformers
import nltk
nltk.download('punkt')
from google.colab import files
import PyPDF2
from nltk.tokenize import sent_tokenize
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, TrainingArguments # Import TrainingArguments from transformers instead of trl
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from trl import SFTTrainer # Keep the import of SFTTrainer from trl
import os
import faiss

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
# Step 2: Upload and Process 7 Books into /content/books
print("Please upload your PDF books.")
os.makedirs('/content/books', exist_ok=True)
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.pdf'):
        os.rename(filename, f'/content/books/{filename}')

book_texts = {}
for filename in os.listdir('/content/books'):
    if filename.endswith('.pdf'):
        with open(f'/content/books/{filename}', 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            text = ""
            for page in reader.pages:
                text += page.extract_text() or ""
            book_texts[filename] = sent_tokenize(text)

# Combine and chunk sentences
all_sentences = []
for text in book_texts.values():
    all_sentences.extend(text)

def chunk_text(sentences, chunk_size=10):
    return [' '.join(sentences[i:i+chunk_size]) for i in range(0, len(sentences), chunk_size)]

chunks = chunk_text(all_sentences)
print(f"Created {len(chunks)} chunks from {len(all_sentences)} sentences.")

Please upload your 8 PDF books.
